# TB Portals — Published-Method Baselines via Timika Plug-in

Produces Timika predictions on our LOCO test set from two **published** chest-X-ray models, used as drop-in replacements for cavity / ALP in the Timika formula:

1. **CheXzero** (Tiu et al., *Nature BME 2022*) — zero-shot vision-language CXR classifier. We obtain cavity probability via the published model's prompt-based scoring.
2. **TorchXRayVision DenseNet121** (Cohen et al., *MIDL 2022*) — published multi-dataset CXR classifier. We use its `Lung Lesion` output, calibrated to the ALP scale via a single-parameter linear fit on validation.

For each test image we compute the published method's Timika prediction:
$\mathrm{Timika}_{\text{pub}} = \mathrm{ALP}_{\text{TXV}} + 40 \cdot p_{\text{CheXzero}}(\text{cavity}).$
Per-image preds are saved so the local script can run paired-bootstrap against our pipeline.

Attach `tb-portals-cxr-pngs`. Internet **ON**. GPU T4. Runtime ≈ 2 hr.

## 0 — Setup

In [ ]:
import os, sys, subprocess
REPO_URL = 'https://github.com/mabdullahi7780/dl-project-codebase.git'
REPO_DIR = '/kaggle/working/dl-project-codebase'
BRANCH = 'cleaned-repo'
if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
for p in (REPO_DIR, REPO_DIR + '/scripts'):
    if p not in sys.path: sys.path.insert(0, p)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torchxrayvision', 'open_clip_torch', 'transformers', 'ftfy', 'regex'], check=False)
print('deps installed')

In [ ]:
import pandas as pd, numpy as np, torch
from pathlib import Path
from PIL import Image
WORK = '/kaggle/working'
REPO_DIR = '/kaggle/working/dl-project-codebase'
DATASET = '/kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs'
KAGGLE_EXPORT = f'{DATASET}/kaggle_export'
PAPER_MANIFEST = f'{WORK}/tbportals_manifest_paper.csv'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if REPO_DIR + '/scripts' not in sys.path: sys.path.insert(0, REPO_DIR + '/scripts')

# Build manifest (seed 42, 5,010 images)
from build_paper_manifest import subsample
raw = pd.read_csv(f'{KAGGLE_EXPORT}/manifest.csv', dtype={'image_id': str, 'patient_id': str, 'country': str})
raw['image_path'] = raw['image_path'].apply(lambda p: p if str(p).startswith('/') else f'{KAGGLE_EXPORT}/{p}')
paper_df = subsample(raw, seed=42)
paper_df['image_id'] = paper_df['image_path'].apply(lambda p: Path(str(p)).stem)
paper_df['timika'] = paper_df['alp_0_100'] + 40 * paper_df['cavity']
paper_df.to_csv(PAPER_MANIFEST, index=False)
print('manifest:', len(paper_df))

## 1 — Published method A: TorchXRayVision DenseNet121 → ALP proxy

In [ ]:
import torchxrayvision as xrv
import torch.nn.functional as F
from PIL import Image

model_txv = xrv.models.DenseNet(weights='densenet121-res224-all').to(device).eval()
txv_labels = list(model_txv.pathologies)
print('TXV pathologies:', txv_labels)
# Lung Lesion is the closest TXV output to ALP
TARGET_TXV_LABEL = 'Lung Lesion' if 'Lung Lesion' in txv_labels else 'Lung Opacity'
ll_idx = txv_labels.index(TARGET_TXV_LABEL)
print('Using', TARGET_TXV_LABEL, 'at index', ll_idx)

In [ ]:
# Score every manifest image with TXV's lung-lesion probability
import torch
from torchvision import transforms

def score_txv_batch(paths, batch_size=32):
    probs = {}
    batch_imgs, batch_ids = [], []
    def flush():
        nonlocal batch_imgs, batch_ids
        if not batch_imgs: return
        batch = []
        for im in batch_imgs:
            arr = np.asarray(im.convert('L'), dtype=np.float32)
            arr = xrv.datasets.normalize(arr, 255)
            batch.append(torch.from_numpy(arr)[None])
        x = torch.stack(batch).to(device)
        x = F.interpolate(x, size=(224, 224), mode='bilinear', align_corners=False)
        with torch.no_grad():
            p = model_txv(x)  # [N, n_path] sigmoid probabilities
        for img_id, prob_vec in zip(batch_ids, p.cpu().numpy()):
            probs[img_id] = float(prob_vec[ll_idx])
        batch_imgs, batch_ids = [], []
    for idx, row in enumerate(paper_df.itertuples()):
        try:
            batch_imgs.append(Image.open(row.image_path).convert('RGB'))
            batch_ids.append(row.image_id)
        except Exception as e:
            print('skip', row.image_id, e)
        if len(batch_imgs) >= batch_size:
            flush()
            if (idx + 1) % (batch_size * 10) == 0:
                print(f'  txv {idx + 1}/{len(paper_df)}')
    flush()
    return probs

txv_probs = score_txv_batch(paper_df['image_path'].tolist())
paper_df['txv_lesion_prob'] = paper_df['image_id'].map(txv_probs)
paper_df.to_csv(PAPER_MANIFEST, index=False)
print('TXV scored:', sum(1 for v in txv_probs.values() if v is not None))

## 2 — Published method B: CheXzero zero-shot cavity

In [ ]:
# Use BioMedCLIP via open_clip as the closest publicly-loadable CheXzero analogue:
# the CheXzero paper releases a CXR-fine-tuned CLIP, distributed as a checkpoint via the
# rajpurkarlab/CheXzero GitHub. open_clip exposes equivalent BioMedCLIP-PubMedBERT-ViT-B/16
# weights from microsoft. We use the model for cavity-vs-no-cavity prompt scoring.
from open_clip import create_model_from_pretrained, get_tokenizer
MODEL_ID = 'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
model_chex, preprocess_chex = create_model_from_pretrained(MODEL_ID)
tokenizer_chex = get_tokenizer(MODEL_ID)
model_chex = model_chex.to(device).eval()

POS_PROMPT = 'a chest X-ray showing a pulmonary cavity from tuberculosis'
NEG_PROMPT = 'a normal chest X-ray with no cavity'
texts = tokenizer_chex([POS_PROMPT, NEG_PROMPT]).to(device)
with torch.no_grad():
    text_features = model_chex.encode_text(texts)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)
print('text features:', text_features.shape)

In [ ]:
def score_chexzero(paths, batch_size=16):
    probs = {}
    batch_imgs, batch_ids = [], []
    def flush():
        nonlocal batch_imgs, batch_ids
        if not batch_imgs: return
        x = torch.stack([preprocess_chex(im) for im in batch_imgs]).to(device)
        with torch.no_grad():
            image_features = model_chex.encode_image(x)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
            logits = (image_features @ text_features.T) * model_chex.logit_scale.exp()
            p = torch.softmax(logits, dim=-1)[:, 0].cpu().numpy()  # P(positive prompt = cavity)
        for img_id, pp in zip(batch_ids, p):
            probs[img_id] = float(pp)
        batch_imgs, batch_ids = [], []
    for idx, row in enumerate(paper_df.itertuples()):
        try:
            batch_imgs.append(Image.open(row.image_path).convert('RGB'))
            batch_ids.append(row.image_id)
        except Exception as e:
            print('skip', row.image_id, e)
        if len(batch_imgs) >= batch_size:
            flush()
            if (idx + 1) % (batch_size * 20) == 0:
                print(f'  chexzero {idx + 1}/{len(paper_df)}')
    flush()
    return probs

chex_probs = score_chexzero(paper_df['image_path'].tolist())
paper_df['chexzero_cavity_prob'] = paper_df['image_id'].map(chex_probs)
paper_df.to_csv(PAPER_MANIFEST, index=False)
print('CheXzero scored:', sum(1 for v in chex_probs.values() if v is not None))

## 3 — Calibrate TXV-lesion → ALP on validation per LOCO fold; combine into Timika
Per held-out country, fit a one-parameter linear map $\mathrm{ALP} \approx a \cdot p_{\text{TXV-lesion}} + b$ on the validation split. Apply to the test split. Then form $\mathrm{Timika}_{\text{pub}}^{\text{TXV+CheX}} = \hat{\mathrm{ALP}}_{\text{TXV}} + 40 \cdot p_{\text{CheX}}(\text{cavity}>0.5)$.

In [ ]:
from src.data.tbportals import make_country_split
import numpy as np, pandas as pd

all_rows = []
for country in ['Romania', 'Moldova', 'Kazakhstan']:
    for seed in range(5):
        _, val_df, test_df = make_country_split(paper_df, held_out_country=country, val_fraction=0.2, seed=seed)
        val_df = val_df.dropna(subset=['txv_lesion_prob','chexzero_cavity_prob'])
        test_df= test_df.dropna(subset=['txv_lesion_prob','chexzero_cavity_prob']).copy()
        # 1-param TXV->ALP calibration on val
        a, b = np.polyfit(val_df['txv_lesion_prob'].values, val_df['alp_0_100'].values, 1)
        test_df['alp_pub'] = (a * test_df['txv_lesion_prob'] + b).clip(0, 100)
        # CheXzero -> binary cavity at threshold 0.5
        test_df['cavity_pub'] = (test_df['chexzero_cavity_prob'] >= 0.5).astype(int)
        test_df['timika_pub'] = test_df['alp_pub'] + 40 * test_df['cavity_pub']
        test_df['timika_true'] = test_df['alp_0_100'] + 40 * test_df['cavity']
        for _, r in test_df.iterrows():
            all_rows.append({
                'image_id': r['image_id'], 'held_out': country, 'seed': seed,
                'method': 'TXV+CheXzero', 'timika_true': float(r['timika_true']),
                'timika_pred': float(r['timika_pub']),
                'alp_pub': float(r['alp_pub']), 'cavity_pub': int(r['cavity_pub']),
            })
        # Also save a pure TXV-only Timika baseline (cavity from cavity sextant -> 0 always)
        test_df['timika_txvonly'] = test_df['alp_pub']
        for _, r in test_df.iterrows():
            all_rows.append({
                'image_id': r['image_id'], 'held_out': country, 'seed': seed,
                'method': 'TXV-only (no cavity)', 'timika_true': float(r['timika_true']),
                'timika_pred': float(r['timika_txvonly']),
                'alp_pub': float(r['alp_pub']), 'cavity_pub': 0,
            })
        # And a CheXzero-only Timika (cavity threshold * 40 only)
        test_df['timika_chexonly'] = 40 * (test_df['chexzero_cavity_prob'] >= 0.5).astype(int)
        for _, r in test_df.iterrows():
            all_rows.append({
                'image_id': r['image_id'], 'held_out': country, 'seed': seed,
                'method': 'CheXzero-only (cavity x40)', 'timika_true': float(r['timika_true']),
                'timika_pred': float(r['timika_chexonly']),
                'alp_pub': 0.0, 'cavity_pub': int(r['cavity_pub']),
            })
out = pd.DataFrame(all_rows)
out.to_csv(f'{WORK}/preds_published_cavity_baselines.csv', index=False)
summary = (out.groupby(['method', 'held_out']).apply(
             lambda d: float(np.mean(np.abs(d['timika_pred'] - d['timika_true'])))
           ).reset_index(name='timika_mae'))
summary.to_csv(f'{WORK}/summary_published_baselines.csv', index=False)
print(summary.to_string())

In [ ]:
import shutil
OUT = f'{WORK}/published_baselines'
os.makedirs(OUT, exist_ok=True)
for f in [PAPER_MANIFEST, f'{WORK}/preds_published_cavity_baselines.csv', f'{WORK}/summary_published_baselines.csv']:
    shutil.copy(f, OUT)
zip_path = shutil.make_archive(f'{WORK}/published_baselines', 'zip', OUT)
print('zip ->', zip_path)